# 4.0 · LSTM en acción — ¿cómo "recuerda" una lluvia?

**Tiempo:** 15-20 min.

**Objetivo:** abrir la caja negra de una LSTM diminuta sobre un dataset de juguete y mirar las **gates** mientras procesan la secuencia.

**Dataset sintético:** la lluvia de hoy provoca una respuesta en el caudal **5 pasos más tarde**. La LSTM tendrá que aprender a *recordar* el pulso de lluvia durante esos 5 pasos antes de actuar.

> Este notebook es un *warm-up* — no usa datos reales. El objetivo es desmitificar la LSTM antes de pasar a `01_lstm_caudal_keras.ipynb`.

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import matplotlib.pyplot as plt

import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Input

plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3})
rng = np.random.default_rng(42)
keras.utils.set_random_seed(0)

## 1 · Dataset sintético — lluvia → caudal con delay de 5 pasos

Construimos una serie de 1.200 puntos:

- **Lluvia** $p_t$: cero salvo en ~10% de los pasos, donde aparece un pulso de magnitud 5.
- **Caudal** $y_t = 1 + 0.5 \cdot p_{t-5} + \varepsilon$: respuesta retardada **5 pasos** + ruido gaussiano pequeño.

In [ ]:
N = 1200
p = np.zeros(N)
rain_idx = rng.choice(N, size=int(N * 0.10), replace=False)
p[rain_idx] = 5.0

y = np.ones(N) + rng.normal(0, 0.05, N)
y[5:] += 0.5 * p[:-5]  # respuesta retardada 5 pasos

# Pinta los primeros 100 pasos para ver el patrón
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 3.5), sharex=True)
sl = slice(0, 100)
ax1.bar(np.arange(N)[sl], p[sl], color="#1f6f8b", alpha=0.7, width=0.7)
ax1.set_ylabel(r"lluvia $p_t$")
ax1.set_title("Dataset sintético — primeros 100 pasos")
ax2.plot(np.arange(N)[sl], y[sl], color="#374151", lw=1.2, marker="o", ms=3)
ax2.set_ylabel(r"caudal $y_t$")
ax2.set_xlabel("paso $t$")
plt.tight_layout()

> **Verifica visualmente** la regla: tras cada pico de lluvia, el caudal sube **5 pasos después** y luego vuelve al baseline. Eso es lo que la LSTM tiene que aprender.

## 2 · Preparar ventanas

Misma estructura que en datos reales: ventana de 15 pasos de **lluvia** como input, caudal del paso siguiente como target. La LSTM **solo ve la lluvia** — tiene que descubrir que la respuesta llega 5 pasos después.

In [ ]:
VENTANA = 15


def crear_ventanas(p, y, ventana):
    X, yt = [], []
    for i in range(len(p) - ventana):
        X.append(p[i : i + ventana])
        yt.append(y[i + ventana])
    return np.asarray(X)[..., None], np.asarray(yt)


X, y_target = crear_ventanas(p, y, VENTANA)
split = int(0.7 * len(X))
X_tr, X_te = X[:split], X[split:]
y_tr, y_te = y_target[:split], y_target[split:]
print(f"X: {X.shape}   train={len(X_tr):,}   test={len(X_te):,}")

## 3 · LSTM diminuta — 4 unidades

Solo **4 unidades** en la capa LSTM. Es suficiente para resolver el problema y permite ver lo que hace cada una.

In [ ]:
model = Sequential(
    [
        Input(shape=(VENTANA, 1)),
        LSTM(4, return_sequences=False),
        Dense(1),
    ]
)
model.compile(optimizer=keras.optimizers.Adam(5e-3), loss="mse")
model.summary()

history = model.fit(
    X_tr,
    y_tr,
    validation_data=(X_te, y_te),
    epochs=30,
    batch_size=32,
    verbose=0,
)

y_pred = model.predict(X_te, verbose=0).flatten()
rmse = float(np.sqrt(((y_te - y_pred) ** 2).mean()))
print(f"RMSE test = {rmse:.3f}   (std baseline = {y_te.std():.3f})")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.0))
ax.plot(y_te[:120], color="black", lw=1, label="real")
ax.plot(y_pred[:120], color="#c2410c", lw=1.2, ls="--", label="LSTM")
ax.set_xlabel("paso (test)")
ax.set_ylabel("caudal")
ax.set_title("Predicción de la LSTM diminuta — primeros 120 puntos del test")
ax.legend()
plt.tight_layout()

## 4 · Abrir la caja — las gates en acción

La capa LSTM de Keras no expone las gates directamente, pero **podemos reconstruirlas a mano** con los pesos aprendidos. Recuerda las ecuaciones:

$$
\begin{aligned}
i_t &= \sigma(W_i x_t + U_i h_{t-1} + b_i) \\
f_t &= \sigma(W_f x_t + U_f h_{t-1} + b_f) \\
\tilde c_t &= \tanh(W_c x_t + U_c h_{t-1} + b_c) \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde c_t \\
o_t &= \sigma(W_o x_t + U_o h_{t-1} + b_o) \\
h_t &= o_t \odot \tanh(c_t)
\end{aligned}
$$

Keras almacena los 4 sets de pesos **concatenados** en el orden `[i, f, c, o]`. Los partimos por gate y ejecutamos el forward pass paso a paso.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


W, U, b = model.layers[0].get_weights()
units = 4
W_i, W_f, W_c, W_o = np.split(W, 4, axis=1)
U_i, U_f, U_c, U_o = np.split(U, 4, axis=1)
b_i, b_f, b_c, b_o = np.split(b, 4)


def lstm_forward_con_gates(x_seq):
    """Ejecuta la LSTM paso a paso y devuelve gates + estados."""
    T = len(x_seq)
    h, c = np.zeros(units), np.zeros(units)
    gates = {k: np.zeros((T, units)) for k in ("i", "f", "o")}
    states = {k: np.zeros((T, units)) for k in ("c", "h")}
    for t in range(T):
        x_t = x_seq[t]
        i = sigmoid(x_t @ W_i + h @ U_i + b_i)
        f = sigmoid(x_t @ W_f + h @ U_f + b_f)
        c = f * c + i * np.tanh(x_t @ W_c + h @ U_c + b_c)
        o = sigmoid(x_t @ W_o + h @ U_o + b_o)
        h = o * np.tanh(c)
        gates["i"][t], gates["f"][t], gates["o"][t] = i, f, o
        states["c"][t], states["h"][t] = c, h
    return gates, states, h

**Sanity check:** la predicción manual (forward pass numpy + capa `Dense`) debe coincidir con la de Keras.

In [ ]:
# Busca una ventana con un único pulso de lluvia hacia el centro
def encontrar_ventana_limpia():
    for i in range(len(X_te)):
        w = X_te[i].flatten()
        nz = np.where(w > 0)[0]
        if len(nz) == 1 and 4 <= nz[0] <= 9:
            return i
    return None


idx = encontrar_ventana_limpia()
sample = X_te[idx]
pos_lluvia = int(np.where(sample.flatten() > 0)[0][0])
print(f"Ventana de test #{idx} — pulso de lluvia en la posición {pos_lluvia}")

gates, states, h_T = lstm_forward_con_gates(sample)

# Sanity: manual h_T → Dense vs Keras predict
W_d, b_d = model.layers[1].get_weights()
manual_pred = float(h_T @ W_d + b_d)
keras_pred = float(model.predict(sample[None, ...], verbose=0).flatten()[0])
print(f"Predicción manual: {manual_pred:.4f}")
print(f"Predicción Keras:  {keras_pred:.4f}")
assert abs(manual_pred - keras_pred) < 1e-4, "El forward manual no coincide con Keras."

## 5 · Visualización — input + 3 gates a lo largo de la ventana

Lo interesante es ver cómo las 3 gates **reaccionan al pulso de lluvia**. Cada línea es una de las 4 unidades.

In [ ]:
fig, axes = plt.subplots(
    4,
    1,
    figsize=(11, 6.5),
    sharex=True,
    gridspec_kw={"hspace": 0.35, "height_ratios": [1, 1, 1, 1]},
)
t_axis = np.arange(VENTANA)

axes[0].bar(t_axis, sample.flatten(), color="#1f6f8b", alpha=0.7, width=0.7)
axes[0].set_ylabel(r"$p_t$")
axes[0].set_title("Input: secuencia de lluvia (pulso aislado)")

for ax, key, title, color in [
    (axes[1], "i", r"Input gate $i_t$ — ¿qué memorizar?", "#7c3aed"),
    (axes[2], "f", r"Forget gate $f_t$ — ¿qué retener del pasado?", "#c2410c"),
    (axes[3], "o", r"Output gate $o_t$ — ¿qué exponer?", "#0e7c5d"),
]:
    for u in range(units):
        ax.plot(t_axis, gates[key][:, u], color=color, alpha=0.55, lw=1.4, label=f"unidad {u}")
    ax.axvline(pos_lluvia, color="#1f6f8b", lw=0.8, ls=":", alpha=0.6)
    ax.set_title(title, fontsize=10)
    ax.set_ylim(-0.05, 1.05)

axes[-1].set_xlabel(r"paso $t$ dentro de la ventana (línea azul = pulso)")
plt.tight_layout()

## 6 · Interpretación

Mira el paso donde está el pulso de lluvia (línea azul vertical):

- **Input gate** sube cerca de 1 en alguna unidad → "memorizo este evento".
- **Forget gate** **baja** en alguna unidad → esa unidad **resetea** su estado anterior para incorporar el nuevo evento (recuerda: $c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t$; cuando $f_t \approx 0$, el pasado se borra y entra solo lo nuevo).
- **Output gate** se mueve para controlar cuánto del estado interno se expone al `Dense` final.

Cada **unidad** se especializa en algo distinto: algunas trackean el evento de lluvia, otras la respuesta de fondo. Esto es **interpretabilidad** real — los gates son **tres regresiones logísticas** decidiendo qué fluye por la celda.

> La "magia" de la LSTM es esto: tres pequeñas sigmoides que aprenden a abrir/cerrar válvulas en el estado interno. Nada más.

## 7 · Bonus — el estado de la celda $c_t$ a lo largo del tiempo

Si las gates son las **válvulas**, el estado de la celda $c_t$ es el **depósito**. Plotea cómo evoluciona durante la ventana:

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.0))
for u in range(units):
    ax.plot(t_axis, states["c"][:, u], lw=1.5, label=f"$c_t$ unidad {u}")
ax.axvline(pos_lluvia, color="#1f6f8b", lw=1.0, ls=":", alpha=0.7, label="pulso de lluvia")
ax.set_xlabel("paso $t$")
ax.set_ylabel(r"$c_t$")
ax.set_title("Estado de la celda — la 'memoria' que pasa de paso a paso")
ax.legend(ncol=3, fontsize=9, loc="lower left")
plt.tight_layout()

Verás que tras el pulso, **alguna unidad cambia bruscamente su $c_t$** y mantiene el nuevo valor durante varios pasos — eso es la "memoria" en acción.

## 8 · Ejercicios cortos

1. **Cambia el delay** a 10 o 15 pasos (`y[5:] += 0.5 * p[:-5]` → `y[10:] += ...`). ¿Sigue funcionando con `VENTANA=15`? Cuándo deja de aprender?
2. **Reduce las unidades** a 1 o 2. ¿Es suficiente? ¿Qué hace la única unidad ahora?
3. **Quita el delay** (respuesta inmediata: `y_t = 1 + 0.5 \cdot p_t`). El problema se vuelve markoviano — ¿cómo cambian las gates?
4. **GRU vs LSTM.** Sustituye `LSTM(4)` por `GRU(4)`. ¿Funciona igual? Las dos gates de GRU (update y reset) son más sutiles de visualizar pero existen.